In [1]:
import cv2
import mediapipe as mp
import time
from datetime import datetime
import numpy as np
from ultralytics import YOLO  

mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
pose = mp_pose.Pose(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

knife_model = YOLO("yolov8_updated.pt")
gun_model = YOLO("yolov8_gun.pt")

def landmark_visible(lm, edges, w, h):
    x = int(lm.x * w)
    y = int(lm.y * h)

    r = 6
    patch = edges[max(0, y-r):min(h, y+r),
                  max(0, x-r):min(w, x+r)]

    if patch.size == 0:
        return False

    return np.mean(patch) > 3

def calculate_face_coverage(face_landmarks, edges, w, h):
    critical_landmarks = [33, 133, 362, 263, 1, 2, 98, 324, 13, 14]
    visible_landmarks = 0

    for idx in critical_landmarks:
        if landmark_visible(face_landmarks[idx], edges, w, h):
            visible_landmarks += 1

    coverage = (visible_landmarks / len(critical_landmarks)) * 100
    return coverage

def analyze_behavior(pose_landmarks):
    left_hand_y = pose_landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y
    right_hand_y = pose_landmarks[mp_pose.PoseLandmark.RIGHT_WRIST].y
    nose_y = pose_landmarks[mp_pose.PoseLandmark.NOSE].y

    if left_hand_y < nose_y or right_hand_y < nose_y:
        return True  
    return False

cap = cv2.VideoCapture(r"C:\Users\Andhavarapu Jahnavi\Desktop\IntelliGuard Multi-Modal AI Threat Detection System\sample.mp4")

alert_message = "Warning: Potential Threat!"
weapon_alert_message = "Weapon Detected!"
alert_duration = 3
last_alert_time = 0

while True:
    success, frame = cap.read()
    if not success:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray_frame, 80, 160)

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(image_rgb)
    pose_results = pose.process(image_rgb)

    h, w = frame.shape[:2]

    unusual_behavior_detected = False

    if face_results.multi_face_landmarks:
        for face_landmarks in face_results.multi_face_landmarks:
            coverage = calculate_face_coverage(
                face_landmarks.landmark, edges, w, h
            )

            print(f"Coverage: {coverage:.2f}%")

            if coverage < 20:
                current_time = time.time()
                if current_time - last_alert_time > alert_duration:
                    cv2.putText(frame, alert_message, (50, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

                    filename = f"warning_image_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                    cv2.imwrite(filename, frame)
                    last_alert_time = current_time

    if pose_results.pose_landmarks:
        unusual_behavior_detected = analyze_behavior(
            pose_results.pose_landmarks.landmark
        )

    if unusual_behavior_detected:
        cv2.putText(frame, "Unusual Behavior Detected!", (50, 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        filename = f"behavior_warning_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
        cv2.imwrite(filename, frame)

    knife_results = knife_model(frame, stream=True)
    gun_results = gun_model(frame, stream=True)

    for result in knife_results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf[0]

            if confidence > 0.25:
                cv2.rectangle(frame, (x1, y1), (x2, y2),
                              (0, 0, 255), 2)

                cv2.putText(frame, f"knife {confidence:.2f}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                            (0, 0, 255), 2)

                cv2.putText(frame, weapon_alert_message, (50, 150),
                            cv2.FONT_HERSHEY_SIMPLEX, 1,
                            (0, 0, 255), 2)

                filename = f"weapon_detected_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                cv2.imwrite(filename, frame)

    for result in gun_results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf[0]

            if confidence > 0.25:
                cv2.rectangle(frame, (x1, y1), (x2, y2),
                              (255, 0, 0), 2)

                cv2.putText(frame, f"gun {confidence:.2f}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                            (255, 0, 0), 2)

                cv2.putText(frame, weapon_alert_message, (50, 150),
                            cv2.FONT_HERSHEY_SIMPLEX, 1,
                            (0, 0, 255), 2)

                filename = f"weapon_detected_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                cv2.imwrite(filename, frame)

    cv2.imshow('Surveillance System', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 384x640 (no detections), 229.9ms
Speed: 10.9ms preprocess, 229.9ms inference, 9.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 knifes, 71.8ms
Speed: 3.8ms preprocess, 71.8ms inference, 8.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 116.0ms
Speed: 8.4ms preprocess, 116.0ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 knifes, 288.8ms
Speed: 6.0ms preprocess, 288.8ms inference, 3.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 202.5ms
Speed: 9.9ms preprocess, 202.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 knifes, 120.8ms
Speed: 4.4ms preprocess, 120.8ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 635.9ms
Speed: 69.9ms preprocess, 635.9ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 knifes, 341.2ms
Speed: 9.4ms preprocess, 341.2ms inference, 1.

In [ ]:
import cv2
import mediapipe as mp
import time
from datetime import datetime
import numpy as np
from ultralytics import YOLO  

# Mediapipe setup
mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
pose = mp_pose.Pose(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# YOLO weapon model

weapon_model = YOLO("yolov8_gun.pt") 

# newly added

def landmark_visible(lm, edges, w, h):
    x = int(lm.x * w)
    y = int(lm.y * h)

    r = 6
    patch = edges[max(0, y-r):min(h, y+r),
                  max(0, x-r):min(w, x+r)]

    if patch.size == 0:
        return False

    return np.mean(patch) > 3

# changed function 
def calculate_face_coverage(face_landmarks, edges, w, h):
    critical_landmarks = [33, 133, 362, 263, 1, 2, 98, 324, 13, 14]
    visible_landmarks = 0

    for idx in critical_landmarks:
        if landmark_visible(face_landmarks[idx], edges, w, h):
            visible_landmarks += 1

    coverage = (visible_landmarks / len(critical_landmarks)) * 100
    return coverage

# Pose behavior logic 
def analyze_behavior(pose_landmarks):
    left_hand_y = pose_landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y
    right_hand_y = pose_landmarks[mp_pose.PoseLandmark.RIGHT_WRIST].y
    nose_y = pose_landmarks[mp_pose.PoseLandmark.NOSE].y

    if left_hand_y < nose_y or right_hand_y < nose_y:
        return True  
    return False

# Webcam
cap = cv2.VideoCapture(0)

alert_message = "Warning: Potential Threat!"
weapon_alert_message = "Weapon Detected!"
alert_duration = 3
last_alert_time = 0

# Main loop

while True:
    success, frame = cap.read()
    if not success:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray_frame, 80, 160)  # New line added

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(image_rgb)
    pose_results = pose.process(image_rgb)

    h, w = frame.shape[:2] # New line added

    unusual_behavior_detected = False

    # Face coverage (NOW CHANGES)

    if face_results.multi_face_landmarks:
        for face_landmarks in face_results.multi_face_landmarks:
            coverage = calculate_face_coverage(
                face_landmarks.landmark, edges, w, h
            )

            print(f"Coverage: {coverage:.2f}%")

            if coverage < 20:
                current_time = time.time()
                if current_time - last_alert_time > alert_duration:
                    cv2.putText(frame, alert_message, (50, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

                    filename = f"warning_image_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                    cv2.imwrite(filename, frame)
                    last_alert_time = current_time

    # ==============================
    # Unusual behavior (UNCHANGED)
    # ==============================
    if pose_results.pose_landmarks:
        unusual_behavior_detected = analyze_behavior(
            pose_results.pose_landmarks.landmark
        )

    if unusual_behavior_detected:
        cv2.putText(frame, "Unusual Behavior Detected!", (50, 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        filename = f"behavior_warning_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
        cv2.imwrite(filename, frame)

    # Weapon detection (UNCHANGED)
    
    weapon_results = weapon_model(frame, stream=True)

    for result in weapon_results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf[0]
            label = result.names[int(box.cls[0])]

            print(label, float(confidence))

            if label in ['pistol'] and confidence > 0.25:
                cv2.rectangle(frame, (x1, y1), (x2, y2),
                              (0, 0, 255), 2)

                cv2.putText(frame, f"{label} {confidence:.2f}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                            (0, 0, 255), 2)

                cv2.putText(frame, weapon_alert_message, (50, 150),
                            cv2.FONT_HERSHEY_SIMPLEX, 1,
                            (0, 0, 255), 2)

                filename = f"weapon_detected_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                cv2.imwrite(filename, frame)

    cv2.imshow('Surveillance System', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [3]:
import cv2
import mediapipe as mp
import time
from datetime import datetime
import numpy as np
from ultralytics import YOLO  

# =============================
# Mediapipe setup
# =============================
mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

pose = mp_pose.Pose(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

hands = mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

# =============================
# YOLO weapon model
# =============================
weapon_model = YOLO("yolov8_updated.pt")

# =============================
# Utility functions (UNCHANGED)
# =============================
def landmark_visible(lm, edges, w, h):
    x = int(lm.x * w)
    y = int(lm.y * h)
    r = 6

    patch = edges[max(0, y-r):min(h, y+r),
                  max(0, x-r):min(w, x+r)]

    if patch.size == 0:
        return False

    return np.mean(patch) > 3


def calculate_face_coverage(face_landmarks, edges, w, h):
    critical_landmarks = [33, 133, 362, 263, 1, 2, 98, 324, 13, 14]
    visible = 0

    for idx in critical_landmarks:
        if landmark_visible(face_landmarks[idx], edges, w, h):
            visible += 1

    return (visible / len(critical_landmarks)) * 100


def analyze_behavior(pose_landmarks):
    left_hand_y = pose_landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y
    right_hand_y = pose_landmarks[mp_pose.PoseLandmark.RIGHT_WRIST].y
    nose_y = pose_landmarks[mp_pose.PoseLandmark.NOSE].y

    return left_hand_y < nose_y or right_hand_y < nose_y


# =============================
# 🆕 Hand interaction utilities
# =============================
def get_hand_bbox(hand_landmarks, w, h):
    xs = [int(lm.x * w) for lm in hand_landmarks.landmark]
    ys = [int(lm.y * h) for lm in hand_landmarks.landmark]
    return min(xs), min(ys), max(xs), max(ys)


def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])

    return inter / (areaA + areaB - inter + 1e-6)


# =============================
# Webcam
# =============================
cap = cv2.VideoCapture(r"C:\Users\Andhavarapu Jahnavi\Desktop\IntelliGuard Multi-Modal AI Threat Detection System\sample.mp4")

alert_message = "Warning: Potential Threat!"
weapon_alert_message = "Weapon Detected!"
alert_duration = 3
last_alert_time = 0

# =============================
# Main loop
# =============================
while True:
    success, frame = cap.read()
    if not success:
        break

    h, w = frame.shape[:2]

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 80, 160)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(rgb)
    pose_results = pose.process(rgb)
    hand_results = hands.process(rgb)

    # =============================
    # Face coverage
    # =============================
    if face_results.multi_face_landmarks:
        for face in face_results.multi_face_landmarks:
            coverage = calculate_face_coverage(face.landmark, edges, w, h)
            print(f"Coverage: {coverage:.2f}%")

            if coverage < 20:
                if time.time() - last_alert_time > alert_duration:
                    cv2.putText(frame, alert_message, (50, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                    cv2.imwrite(
                        f"warning_{datetime.now().strftime('%H-%M-%S')}.jpg", frame
                    )
                    last_alert_time = time.time()


    # Pose behavior
    # =============================
    if pose_results.pose_landmarks:
        if analyze_behavior(pose_results.pose_landmarks.landmark):
            cv2.putText(frame, "Unusual Behavior Detected!", (50, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    # =============================
    # Hand bounding boxes
    # =============================
    hand_boxes = []
    if hand_results.multi_hand_landmarks:
        for hand in hand_results.multi_hand_landmarks:
            box = get_hand_bbox(hand, w, h)
            hand_boxes.append(box)
            cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]),
                          (255, 255, 0), 2)

    # =============================
    # Weapon detection + interaction
    # =============================
    weapon_results = weapon_model(frame, stream=True)

    for result in weapon_results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            label = result.names[int(box.cls[0])]
            conf = float(box.conf[0])

            if label in ['knife', 'pistol'] and conf > 0.25:
                weapon_box = (x1, y1, x2, y2)

                weapon_in_hand = False
                for hb in hand_boxes:
                    if iou(weapon_box, hb) > 0.15:
                        weapon_in_hand = True
                        break

                color = (0, 0, 255) if weapon_in_hand else (0, 255, 255)
                text = "Weapon in Hand" if weapon_in_hand else "Weapon Detected"

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"{text} {conf:.2f}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    cv2.imshow("Surveillance System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 384x640 (no detections), 55.8ms
Speed: 17.5ms preprocess, 55.8ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 66.9ms
Speed: 2.1ms preprocess, 66.9ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 54.1ms
Speed: 1.6ms preprocess, 54.1ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 44.4ms
Speed: 2.1ms preprocess, 44.4ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 50.6ms
Speed: 1.9ms preprocess, 50.6ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 47.6ms
Speed: 1.5ms preprocess, 47.6ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 47.2ms
Speed: 1.7ms preprocess, 47.2ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 55.6ms
Speed: 1.7ms preprocess, 55.6ms 

In [ ]:
import cv2
import mediapipe as mp
import time
from datetime import datetime
import numpy as np
from ultralytics import YOLO  

# =============================
# Mediapipe setup
# =============================
mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

pose = mp_pose.Pose(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

hands = mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

# =============================
# YOLO weapon model
# =============================
weapon_model = YOLO("yolov8_updated.pt")

# =============================
# Utility functions (UNCHANGED)
# =============================
def landmark_visible(lm, edges, w, h):
    x = int(lm.x * w)
    y = int(lm.y * h)
    r = 6

    patch = edges[max(0, y-r):min(h, y+r),
                  max(0, x-r):min(w, x+r)]

    if patch.size == 0:
        return False

    return np.mean(patch) > 3


def calculate_face_coverage(face_landmarks, edges, w, h):
    critical_landmarks = [33, 133, 362, 263, 1, 2, 98, 324, 13, 14]
    visible = 0

    for idx in critical_landmarks:
        if landmark_visible(face_landmarks[idx], edges, w, h):
            visible += 1

    return (visible / len(critical_landmarks)) * 100


def analyze_behavior(pose_landmarks):
    left_hand_y = pose_landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y
    right_hand_y = pose_landmarks[mp_pose.PoseLandmark.RIGHT_WRIST].y
    nose_y = pose_landmarks[mp_pose.PoseLandmark.NOSE].y

    return left_hand_y < nose_y or right_hand_y < nose_y


# =============================
# Hand interaction utilities
# =============================
def get_hand_bbox(hand_landmarks, w, h):
    xs = [int(lm.x * w) for lm in hand_landmarks.landmark]
    ys = [int(lm.y * h) for lm in hand_landmarks.landmark]
    return min(xs), min(ys), max(xs), max(ys)


def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])

    return inter / (areaA + areaB - inter + 1e-6)


# =============================
# Webcam
# =============================
cap = cv2.VideoCapture(0)

alert_message = "Warning: Potential Threat!"
weapon_alert_message = "Weapon in Hand!"
alert_duration = 3
last_alert_time = 0

# 🆕 screenshot control
last_capture_time = 0
capture_cooldown = 3  # seconds

# Main loop
while True:
    success, frame = cap.read()
    if not success:
        break

    h, w = frame.shape[:2]

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 80, 160)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(rgb)
    pose_results = pose.process(rgb)
    hand_results = hands.process(rgb)

    # Face coverage
    if face_results.multi_face_landmarks:
        for face in face_results.multi_face_landmarks:
            coverage = calculate_face_coverage(face.landmark, edges, w, h)
            print(f"Coverage: {coverage:.2f}%")

            if coverage < 20:
                if time.time() - last_alert_time > alert_duration:
                    cv2.putText(frame, alert_message, (50, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                    last_alert_time = time.time()

    # Pose behavior
    if pose_results.pose_landmarks:
        if analyze_behavior(pose_results.pose_landmarks.landmark):
            cv2.putText(frame, "Unusual Behavior Detected!", (50, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    # Hand bounding boxes
    hand_boxes = []
    if hand_results.multi_hand_landmarks:
        for hand in hand_results.multi_hand_landmarks:
            box = get_hand_bbox(hand, w, h)
            hand_boxes.append(box)
            cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]),
                          (255, 255, 0), 2)

    # Weapon detection + screenshot
    weapon_results = weapon_model(frame, stream=True)

    for result in weapon_results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            label = result.names[int(box.cls[0])]
            conf = float(box.conf[0])

            if label in ['knife','gun', 'pistol'] and conf > 0.25:
                weapon_box = (x1, y1, x2, y2)

                weapon_in_hand = False
                for hb in hand_boxes:
                    if iou(weapon_box, hb) > 0.15:
                        weapon_in_hand = True
                        break

                if weapon_in_hand:
                    cv2.rectangle(frame, (x1, y1), (x2, y2),
                                  (0, 0, 255), 2)
                    cv2.putText(frame, weapon_alert_message,
                                (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                                (0, 0, 255), 2)

                    # 🆕 Screenshot capture
                    if time.time() - last_capture_time > capture_cooldown:
                        filename = f"weapon_in_hand_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                        cv2.imwrite(filename, frame)
                        print(f"[SAVED] {filename}")
                        last_capture_time = time.time()
                else:
                    cv2.rectangle(frame, (x1, y1), (x2, y2),
                                  (0, 255, 255), 2)

    cv2.imshow("Surveillance System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 480x640 (no detections), 188.3ms
Speed: 8.9ms preprocess, 188.3ms inference, 7.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 206.6ms
Speed: 5.9ms preprocess, 206.6ms inference, 4.3ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 0.00%

0: 480x640 (no detections), 232.7ms
Speed: 6.9ms preprocess, 232.7ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 30.00%

0: 480x640 1 knife, 164.6ms
Speed: 2.8ms preprocess, 164.6ms inference, 8.8ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 10.00%

0: 480x640 (no detections), 177.5ms
Speed: 4.5ms preprocess, 177.5ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 0.00%

0: 480x640 1 knife, 141.7ms
Speed: 2.3ms preprocess, 141.7ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 10.00%

0: 480x640 1 knife, 158.9ms
Speed: 3.1ms preprocess, 158.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 6

In [2]:
import cv2
import mediapipe as mp
import time
from datetime import datetime
import numpy as np
from ultralytics import YOLO  
from collections import deque, defaultdict

# =============================
# Mediapipe setup - FIXED: Added refine_landmarks=True
# =============================
mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
    refine_landmarks=True  # CRITICAL: This enables iris landmarks (468-477)
)

pose = mp_pose.Pose(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

hands = mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

# =============================
# YOLO weapon model
# =============================
weapon_model = YOLO("yolov8_updated.pt")

# =============================
# Utility functions (UNCHANGED)
# =============================
def landmark_visible(lm, edges, w, h):
    x = int(lm.x * w)
    y = int(lm.y * h)
    r = 6

    patch = edges[max(0, y-r):min(h, y+r),
                  max(0, x-r):min(w, x+r)]

    if patch.size == 0:
        return False

    return np.mean(patch) > 3


def calculate_face_coverage(face_landmarks, edges, w, h):
    critical_landmarks = [33, 133, 362, 263, 1, 2, 98, 324, 13, 14]
    visible = 0

    for idx in critical_landmarks:
        if landmark_visible(face_landmarks[idx], edges, w, h):
            visible += 1

    return (visible / len(critical_landmarks)) * 100


def analyze_behavior(pose_landmarks):
    left_hand_y = pose_landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y
    right_hand_y = pose_landmarks[mp_pose.PoseLandmark.RIGHT_WRIST].y
    nose_y = pose_landmarks[mp_pose.PoseLandmark.NOSE].y

    return left_hand_y < nose_y or right_hand_y < nose_y


# =============================
# Hand interaction utilities
# =============================
def get_hand_bbox(hand_landmarks, w, h):
    xs = [int(lm.x * w) for lm in hand_landmarks.landmark]
    ys = [int(lm.y * h) for lm in hand_landmarks.landmark]
    return min(xs), min(ys), max(xs), max(ys)


def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])

    return inter / (areaA + areaB - inter + 1e-6)


# =============================
# FIXED: Gaze Detection with fallback
# =============================
def estimate_gaze_direction(face_landmarks, w, h):
    """
    Detect if person is looking at camera, away, or scanning area
    """
    # Check if iris landmarks are available (requires refine_landmarks=True)
    if len(face_landmarks) < 478:
        # Fallback: Use eye corner positions instead
        # Left eye: outer corner vs inner corner
        left_eye_outer = face_landmarks[33]
        left_eye_inner = face_landmarks[133]
        
        # Right eye
        right_eye_outer = face_landmarks[362]
        right_eye_inner = face_landmarks[263]
        
        # Approximate by checking pupil position relative to eye width
        # This is less accurate but works without iris landmarks
        left_center_x = (left_eye_outer.x + left_eye_inner.x) / 2
        right_center_x = (right_eye_outer.x + right_eye_inner.x) / 2
        
        # Use simple heuristic based on nose position
        nose = face_landmarks[1]
        
        # If nose is centered between eyes, assume forward gaze
        eye_center = (left_center_x + right_center_x) / 2
        
        if abs(nose.x - eye_center) < 0.02:
            return "DIRECT_STARE"
        elif nose.x < eye_center - 0.02:
            return "LOOKING_RIGHT"
        elif nose.x > eye_center + 0.02:
            return "LOOKING_LEFT"
        else:
            return "NORMAL"
    
    # Original code with iris landmarks
    left_eye_center = face_landmarks[468]  # Iris landmark
    left_eye_outer = face_landmarks[33]
    left_eye_inner = face_landmarks[133]
    
    right_eye_center = face_landmarks[473]
    right_eye_outer = face_landmarks[362]
    right_eye_inner = face_landmarks[263]
    
    left_ratio = (left_eye_center.x - left_eye_outer.x) / (left_eye_inner.x - left_eye_outer.x)
    right_ratio = (right_eye_center.x - right_eye_outer.x) / (right_eye_inner.x - right_eye_outer.x)
    
    avg_ratio = (left_ratio + right_ratio) / 2
    
    if 0.4 < avg_ratio < 0.6:
        return "DIRECT_STARE"
    elif avg_ratio < 0.3:
        return "LOOKING_LEFT"
    elif avg_ratio > 0.7:
        return "LOOKING_RIGHT"
    else:
        return "NORMAL"


def get_head_pose(face_landmarks, w, h):
    """
    Estimate 3D head orientation - are they trying to hide face?
    """
    # 2D image points
    image_points = np.array([
        (face_landmarks[1].x * w, face_landmarks[1].y * h),      # Nose tip
        (face_landmarks[152].x * w, face_landmarks[152].y * h),  # Chin
        (face_landmarks[33].x * w, face_landmarks[33].y * h),    # Left eye
        (face_landmarks[263].x * w, face_landmarks[263].y * h),  # Right eye
        (face_landmarks[61].x * w, face_landmarks[61].y * h),    # Left mouth
        (face_landmarks[291].x * w, face_landmarks[291].y * h)   # Right mouth
    ], dtype="double")
    
    # 3D model points
    model_points = np.array([
        (0.0, 0.0, 0.0),
        (0.0, -330.0, -65.0),
        (-225.0, 170.0, -135.0),
        (225.0, 170.0, -135.0),
        (-150.0, -150.0, -125.0),
        (150.0, -150.0, -125.0)
    ])
    
    # Camera internals
    focal_length = w
    center = (w/2, h/2)
    camera_matrix = np.array([
        [focal_length, 0, center[0]],
        [0, focal_length, center[1]],
        [0, 0, 1]
    ], dtype="double")
    
    dist_coeffs = np.zeros((4,1))
    
    success, rotation_vec, translation_vec = cv2.solvePnP(
        model_points, image_points, camera_matrix, dist_coeffs,
        flags=cv2.SOLVEPNP_ITERATIVE
    )
    
    # Convert rotation vector to euler angles
    rotation_mat, _ = cv2.Rodrigues(rotation_vec)
    pose_mat = cv2.hconcat((rotation_mat, translation_vec))
    _, _, _, _, _, _, euler_angles = cv2.decomposeProjectionMatrix(pose_mat)
    
    pitch, yaw, roll = euler_angles.flatten()[:3]
    
    # Interpret angles
    if abs(yaw) > 25:
        return "HEAD_TURNED_AWAY"
    if pitch > 15:
        return "LOOKING_DOWN"
    if pitch < -15:
        return "LOOKING_UP"
    
    return "NORMAL"


# =============================
# Temporal Tracking for Gaze
# =============================
class GazeTracker:
    def __init__(self, history_length=30):
        self.gaze_history = deque(maxlen=history_length)
        self.pose_history = deque(maxlen=history_length)
        self.direct_stare_start_time = None
        self.stare_threshold = 2.0
        
    def update(self, gaze, head_pose):
        self.gaze_history.append(gaze)
        self.pose_history.append(head_pose)
        
        if len(self.gaze_history) < 15:
            return False, 0.0
        
        recent_stares = sum(1 for g in self.gaze_history if g == "DIRECT_STARE")
        recent_normal_pose = sum(1 for p in self.pose_history if p == "NORMAL")
        
        stare_ratio = recent_stares / len(self.gaze_history)
        pose_ratio = recent_normal_pose / len(self.pose_history)
        
        if stare_ratio > 0.7 and pose_ratio > 0.7:
            if self.direct_stare_start_time is None:
                self.direct_stare_start_time = time.time()
            
            stare_duration = time.time() - self.direct_stare_start_time
            
            if stare_duration > self.stare_threshold:
                return True, stare_duration
        else:
            self.direct_stare_start_time = None
        
        return False, 0.0
    
    def reset(self):
        self.gaze_history.clear()
        self.pose_history.clear()
        self.direct_stare_start_time = None


# =============================
# Webcam
# =============================
cap = cv2.VideoCapture(0)

alert_message = "Warning: Potential Threat!"
weapon_alert_message = "Weapon in Hand!"
alert_duration = 3
last_alert_time = 0

last_capture_time = 0
capture_cooldown = 3

threat_score = 0

gaze_tracker = GazeTracker(history_length=30)

# Main loop
while True:
    success, frame = cap.read()
    if not success:
        break

    h, w = frame.shape[:2]

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 80, 160)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(rgb)
    pose_results = pose.process(rgb)
    hand_results = hands.process(rgb)

    threat_score = 0

    if face_results.multi_face_landmarks:
        for face in face_results.multi_face_landmarks:
            coverage = calculate_face_coverage(face.landmark, edges, w, h)
            print(f"Coverage: {coverage:.2f}%")

            if coverage < 20:
                if time.time() - last_alert_time > alert_duration:
                    cv2.putText(frame, alert_message, (50, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                    last_alert_time = time.time()

            gaze = estimate_gaze_direction(face.landmark, w, h)
            head_pose = get_head_pose(face.landmark, w, h)
            
            is_prolonged_stare, stare_duration = gaze_tracker.update(gaze, head_pose)
            
            if is_prolonged_stare:
                cv2.putText(frame, f"Prolonged Stare: {stare_duration:.1f}s", (50, 250),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
                threat_score += 15
            
            cv2.putText(frame, f"Gaze: {gaze} | Pose: {head_pose}", (50, 280),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    else:
        gaze_tracker.reset()

    if pose_results.pose_landmarks:
        if analyze_behavior(pose_results.pose_landmarks.landmark):
            cv2.putText(frame, "Unusual Behavior Detected!", (50, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    hand_boxes = []
    if hand_results.multi_hand_landmarks:
        for hand in hand_results.multi_hand_landmarks:
            box = get_hand_bbox(hand, w, h)
            hand_boxes.append(box)
            cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]),
                          (255, 255, 0), 2)

    weapon_results = weapon_model(frame, stream=True)

    for result in weapon_results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            label = result.names[int(box.cls[0])]
            conf = float(box.conf[0])

            if label in ['knife','gun', 'pistol'] and conf > 0.25:
                weapon_box = (x1, y1, x2, y2)

                weapon_in_hand = False
                for hb in hand_boxes:
                    if iou(weapon_box, hb) > 0.15:
                        weapon_in_hand = True
                        break

                if weapon_in_hand:
                    cv2.rectangle(frame, (x1, y1), (x2, y2),
                                  (0, 0, 255), 2)
                    cv2.putText(frame, weapon_alert_message,
                                (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                                (0, 0, 255), 2)

                    if time.time() - last_capture_time > capture_cooldown:
                        filename = f"weapon_in_hand_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                        cv2.imwrite(filename, frame)
                        print(f"[SAVED] {filename}")
                        last_capture_time = time.time()
                else:
                    cv2.rectangle(frame, (x1, y1), (x2, y2),
                                  (0, 255, 255), 2)

    cv2.imshow("Surveillance System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Coverage: 10.00%

0: 480x640 (no detections), 174.7ms
Speed: 11.3ms preprocess, 174.7ms inference, 10.6ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 20.00%

0: 480x640 (no detections), 198.8ms
Speed: 6.0ms preprocess, 198.8ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 0.00%

0: 480x640 (no detections), 180.3ms
Speed: 3.2ms preprocess, 180.3ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 10.00%

0: 480x640 (no detections), 155.9ms
Speed: 4.0ms preprocess, 155.9ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 10.00%

0: 480x640 (no detections), 154.3ms
Speed: 3.6ms preprocess, 154.3ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 0.00%

0: 480x640 (no detections), 156.8ms
Speed: 3.0ms preprocess, 156.8ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)
Coverage: 0.00%

0: 480x640 (no detections), 150.4ms
Speed: 3.1ms preprocess, 150.4ms in